# Model Tuning

This round explores focused configurations of Random Forest, Extra Trees, and boosting models. The goal is to improve macro F1 and recall for `functional needs repair`, rather than maximizing accuracy alone.

In [2]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.metrics import recall_score

PROJECT_ROOT = None
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').is_dir() and (candidate / 'data').is_dir():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root containing src/ and data/.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_pump_it_up_data
from src.modeling import build_classifier_pipeline, classification_metrics, stratified_split
from scripts.tune_models import model_configs

train_values, train_labels, _ = load_pump_it_up_data(PROJECT_ROOT / 'data')
train = train_values.merge(train_labels, on='id', validate='one_to_one')
X = train.drop(columns=['status_group'])
y = train['status_group']
X_train, X_validation, y_train, y_validation = stratified_split(X, y)

In [ ]:
rows = []
for name, estimator in model_configs().items():
    pipeline = build_classifier_pipeline(X_train, estimator)
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_validation)
    metrics = classification_metrics(y_validation, predictions)
    rows.append({
        'model': name,
        'accuracy': metrics['accuracy'],
        'macro_f1': metrics['macro_f1'],
        'repair_recall': recall_score(
            y_validation, predictions, labels=['functional needs repair'], average=None
        )[0],
    })

results = pd.DataFrame(rows).sort_values('macro_f1', ascending=False).round(4)
display(results)

## Provisional decision

The `rf_balanced_leaf2` configuration provides the best macro F1, while `extra_trees_balanced_leaf2` provides the best minority-class recall. The next decision should incorporate error costs and, before closing model selection, use cross-validation to reduce dependence on a single split.